# Hafta 3 — Klasik Metin Temsilleri

Gömmelerden önce metin, sayım vektörlerinden ibaretti. Bu modeller küçük verilerde ve kısa belgelerde hâlâ kazanır. Hepsini sıfırdan kuruyoruz.

## Öğrenme Hedefleri

- Sözcük Torbası (Bag-of-Words), TF-IDF ve n-gram modellerini saf NumPy ile uygulamak.
- TF-IDF ağırlıklandırmasını bilgi kuramı ilkelerinden türetmek.
- Gizli Anlamsal Analizi (LSA) kesilmiş SVD ile uygulamak.
- Eksiksiz bir seyrek vektör metin sınıflandırıcısı kurmak ve gerçek bir veri kümesi üzerinde kıyaslamak.

## Zorunlu Okumalar

- Salton, G., Wong, A. ve Yang, C. S. (1975). *A Vector Space Model for Automatic Indexing*.
- Deerwester, S. vd. (1990). *Indexing by Latent Semantic Analysis*.
- Chen, S. F. ve Goodman, J. (1999). *An Empirical Study of Smoothing Techniques for Language Modeling*.

In [ ]:
import sys, re
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

np.random.seed(0)

def tokenize(text):
    return re.findall(r"\w+", text.lower(), flags=re.UNICODE)

## 1. Vektör Uzayı Modeli

Salton'ın (1975) içgörüsü: her belgeyi, sözlükteki her terim için bir boyut olmak üzere $\mathbb{R}^{|V|}$ uzayında bir vektör olarak temsil et. Belgeler arasındaki benzerlik kosinüstür. Klasik bilgi erişiminin bütün yapısı bu tek fikre dayanır.

Sonraki her adımı somut sayılara dayandırmak için çok küçük bir derlem kuruyoruz (örnek cümleler İngilizcedir).

In [ ]:
CORPUS = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are pets",
    "machine learning models process text",
    "natural language processing uses text models",
    "deep learning is a subset of machine learning",
    "the cat chased the mouse",
    "the dog barked at the cat",
]
LABELS = ['hayvan', 'hayvan', 'hayvan', 'teknoloji', 'teknoloji', 'teknoloji', 'hayvan', 'hayvan']

tokenized = [tokenize(d) for d in CORPUS]
vocab = sorted(set(tok for doc in tokenized for tok in doc))
tok2id = {t: i for i, t in enumerate(vocab)}
print(f"Sözlük ({len(vocab)} terim): {vocab}")

## 2. Sözcük Torbası (Bag-of-Words)

$\mathbf{x}_d \in \mathbb{N}^{|V|}$; burada $x_{d,t}$, $t$ teriminin $d$ belgesindeki sayısıdır. Pratikte en iyisi seyrek bir matris olarak saklamaktır; açıklık için yoğun NumPy kullanıyoruz.

In [ ]:
def bag_of_words(docs, vocab):
    tok2id = {t: i for i, t in enumerate(vocab)}
    X = np.zeros((len(docs), len(vocab)), dtype=np.float64)
    for d, doc in enumerate(docs):
        for tok in doc:
            if tok in tok2id:
                X[d, tok2id[tok]] += 1
    return X

X_bow = bag_of_words(tokenized, vocab)
print(f"X_bow boyutu: {X_bow.shape}")
print(f"Belge 0 ({CORPUS[0]!r}):")
for t, count in zip(vocab, X_bow[0]):
    if count > 0:
        print(f"  {t!r:20s} {int(count)}")

## 3. TF-IDF — bilgi kuramından türetme

**Terim sıklığı**: $\text{tf}(t, d) = $ $t$'nin $d$ içindeki sayısı; bazen logaritmik ölçeklenir: $1 + \log \text{tf}$.

**Ters belge sıklığı**:

$$\text{idf}(t) = \log \frac{N}{\text{df}(t)}, \quad \text{df}(t) = |\{d : t \in d\}|.$$

Bu, düzgün olarak seçilmiş bir belgede $t$'yi gözlemlemenin şaşırtıcılığı — yani bilgi içeriği — dir. *Her* belgede geçen bir terimin $\text{idf} = 0$'dır ve ayırt edici hiçbir bilgi taşımaz. Tek bir belgede geçen bir terimin $\text{idf} = \log N$'dir — en yüksek özgüllük.

**TF-IDF** ağırlığı $\text{tf}(t, d) \cdot \text{idf}(t)$'dir. Belgeler genellikle L2-normalize edilir; böylece kosinüs benzerliği bir nokta çarpımına indirgenir.

In [ ]:
def tfidf(docs_tokens, vocab):
    N = len(docs_tokens)
    X = bag_of_words(docs_tokens, vocab)
    # logaritmik ölçekli tf
    tf = np.where(X > 0, 1 + np.log(X + 1e-12), 0.0)
    # df
    df = (X > 0).sum(axis=0)
    idf = np.log((N + 1) / (df + 1)) + 1  # yumuşatılmış (sklearn'ün smooth_idf kuralı)
    W = tf * idf
    # satırları L2-normalize et
    norms = np.linalg.norm(W, axis=1, keepdims=True) + 1e-12
    return W / norms, idf

X_tfidf, idf_vec = tfidf(tokenized, vocab)
print("Terim başına IDF (azalan sırada):")
for t, w in sorted(zip(vocab, idf_vec), key=lambda kv: -kv[1])[:10]:
    print(f"  {t!r:20s} idf = {w:.3f}")
print(f"\n0. satırın normu: {np.linalg.norm(X_tfidf[0]):.4f} (1.0 olmalı)")

In [ ]:
# Kosinüs benzerlik matrisi — köşegen 1.0'dır, köşegen dışı konu yapısını ortaya çıkarır.
sim = X_tfidf @ X_tfidf.T
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim, cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(range(len(CORPUS))); ax.set_yticks(range(len(CORPUS)))
ax.set_xticklabels([f"d{i}" for i in range(len(CORPUS))])
ax.set_yticklabels([f"d{i}: {c[:30]}" for i, c in enumerate(CORPUS)])
for i in range(len(CORPUS)):
    for j in range(len(CORPUS)):
        ax.text(j, i, f"{sim[i,j]:.2f}", ha='center', va='center',
                color='white' if sim[i,j] < 0.5 else 'black', fontsize=8)
ax.set_title('Belge–belge kosinüs benzerliği (TF-IDF)')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 4. N-gram dil modelleri

En çok olabilirlikli bir bigram modeli şunu tahmin eder:

$$P(w_t \mid w_{t-1}) = \frac{c(w_{t-1}, w_t)}{c(w_{t-1})}.$$

Ancak birçok bigram eğitim verisinde sıfır kez geçer — bunlara 0 olasılık atamak, onları içeren her ayrılmış metinde sonsuz perpleksite verir. Yumuşatma bunu düzeltir. İki yöntem uyguluyoruz: Laplace (bir ekleme) ve daha gelişmiş Kneser–Ney.

Kneser–Ney'de bağlam sözcüğü eğitimde hiç görülmemişse ($c(w_{t-1}) = 0$) model tamamen devam olasılığına $P_{\text{devam}}(w)$ geri çekilir; hiç görülmemiş sözcükler için de $1/|V|$'lik bir taban kullanılır. Böylece hiçbir olasılık sıfır olmaz.

In [ ]:
class BigramLM:
    def __init__(self, smoothing='laplace', discount=0.75):
        self.smoothing = smoothing
        self.discount = discount

    def fit(self, docs_tokens):
        self.unigram = Counter()
        self.bigram = Counter()
        self.context = Counter()
        for doc in docs_tokens:
            doc = ['<s>'] + doc + ['</s>']
            for i in range(len(doc)):
                self.unigram[doc[i]] += 1
                if i > 0:
                    self.bigram[(doc[i-1], doc[i])] += 1
                    self.context[doc[i-1]] += 1
        self.V = len(self.unigram)
        # Kneser–Ney devamı için: P_devam(w) ∝ w'den önce gelen farklı bağlamların sayısı.
        self.preceding = defaultdict(set)
        for (a, b) in self.bigram:
            self.preceding[b].add(a)
        self.total_bigrams = sum(self.bigram.values())
        return self

    def prob(self, w_prev, w):
        c_ab = self.bigram[(w_prev, w)]
        c_a = self.context[w_prev]
        if self.smoothing == 'laplace':
            return (c_ab + 1) / (c_a + self.V)
        elif self.smoothing == 'kneser_ney':
            d = self.discount
            # Devam olasılığı: w'nin ne sıklıkla yeni bir devam olarak göründüğü.
            p_cont = len(self.preceding[w]) / max(len(self.bigram), 1)
            if p_cont == 0:
                p_cont = 1.0 / self.V  # hiç görülmemiş sözcük: düzgün taban
            if c_a == 0:
                return p_cont  # hiç görülmemiş bağlam: tamamen devam olasılığına geri çekil
            n_followers = sum(1 for (a, _) in self.bigram if a == w_prev)
            lam = (d * n_followers) / c_a
            return max(c_ab - d, 0) / c_a + lam * p_cont
        else:
            raise ValueError(self.smoothing)

    def perplexity(self, docs_tokens):
        log_p, n = 0.0, 0
        for doc in docs_tokens:
            doc = ['<s>'] + doc + ['</s>']
            for i in range(1, len(doc)):
                p = self.prob(doc[i-1], doc[i])
                log_p += np.log(max(p, 1e-12))
                n += 1
        return float(np.exp(-log_p / n))

lm_lap = BigramLM('laplace').fit(tokenized)
lm_kn  = BigramLM('kneser_ney').fit(tokenized)
print(f"Laplace      perpleksite (eğitim kümesi): {lm_lap.perplexity(tokenized):.2f}")
print(f"Kneser–Ney   perpleksite (eğitim kümesi): {lm_kn.perplexity(tokenized):.2f}")

# Görülmüş ve görülmemiş bigramları karıştıran ayrılmış bir cümle:
test = [['the', 'cat', 'barked', 'at', 'the', 'mouse']]
print("\nAyrılmış cümlede perpleksite (yeni/görülmüş karışık):")
print(f"  Laplace    : {lm_lap.perplexity(test):.2f}")
print(f"  Kneser–Ney : {lm_kn.perplexity(test):.2f}")

# Hiç görülmemiş bir bağlam sözcüğü ('zebra') — Kneser–Ney sıfır olasılık vermemeli:
test_unseen = [['the', 'zebra', 'sat', 'on', 'the', 'mat']]
print("\nGörülmemiş bağlam içeren cümlede perpleksite:")
print(f"  Laplace    : {lm_lap.perplexity(test_unseen):.2f}")
print(f"  Kneser–Ney : {lm_kn.perplexity(test_unseen):.2f}")

## 5. Gizli Anlamsal Analiz (LSA)

LSA, TF-IDF matrisi üzerinde kesilmiş SVD'dir:

$$X \approx U_k \Sigma_k V_k^\top.$$

$U_k$'nın satırları belge gömmeleri, $V_k$'nın satırları terim gömmeleridir; ikisi de $\mathbb{R}^k$ içindedir. Öndeki tekil vektörler eş-oluşum yapısını yakalar — yalnızca ikinci dereceden istatistiklerden ortaya çıkan konular. Bu, word2vec'in (Hafta 4) kavramsal atasıdır.

In [ ]:
def lsa(X, k=2):
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    return U[:, :k] * s[:k], Vt[:k, :].T, s

doc_emb, term_emb, sing = lsa(X_tfidf, k=2)
print(f"Tekil değerler: {sing}")
print(f"Belge gömmesi boyutu: {doc_emb.shape}")
print(f"Terim gömmesi boyutu: {term_emb.shape}")

# Belgeleri 2B LSA uzayında görselleştir — konu kümeleri ortaya çıkmalı.
fig, ax = plt.subplots(figsize=(9, 6))
colors = ['tab:blue' if y == 'hayvan' else 'tab:red' for y in LABELS]
ax.scatter(doc_emb[:, 0], doc_emb[:, 1], c=colors, s=100, edgecolors='black')
for i, (x, y) in enumerate(doc_emb):
    ax.annotate(f"d{i}", (x, y), xytext=(5, 5), textcoords='offset points', fontsize=10)
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set(xlabel='LSA boyut 1', ylabel='LSA boyut 2',
       title='2B LSA uzayında belgeler (mavi=hayvan, kırmızı=teknoloji)')
plt.tight_layout(); plt.show()

## 6. Eksiksiz bir sınıflandırıcı: TF-IDF üzerinde lojistik regresyon

TF-IDF öznitelikleri üzerinde bir lojistik regresyon sınıflandırıcısı güçlü bir taban çizgisidir. Onu toplu gradyan inişiyle sıfırdan uyguluyoruz — sklearn yok — ve ardından doğruluyoruz.

In [ ]:
def logreg_fit(X, y, lr=0.5, n_iter=500, l2=0.01):
    n, d = X.shape
    w = np.zeros(d); b = 0.0
    for _ in range(n_iter):
        z = X @ w + b
        p = 1 / (1 + np.exp(-z))
        grad_w = X.T @ (p - y) / n + l2 * w
        grad_b = (p - y).mean()
        w -= lr * grad_w; b -= lr * grad_b
    return w, b

def logreg_predict(X, w, b):
    return (1 / (1 + np.exp(-(X @ w + b)))) > 0.5

y = np.array([1 if l == 'teknoloji' else 0 for l in LABELS])
w, b = logreg_fit(X_tfidf, y, n_iter=2000)
preds = logreg_predict(X_tfidf, w, b)
acc = (preds == y).mean()
print(f"Eğitim doğruluğu: {acc:.3f}")

# Hangi terimler 'teknoloji' sınıfını en güçlü biçimde öngörüyor?
top = np.argsort(w)[::-1][:5]
print("\n'teknoloji' için en belirleyici terimler:")
for i in top:
    print(f"  {vocab[i]!r:20s} ağırlık = {w[i]:+.3f}")
print("\n'hayvan' için en belirleyici terimler:")
for i in np.argsort(w)[:5]:
    print(f"  {vocab[i]!r:20s} ağırlık = {w[i]:+.3f}")

## 7. Alıştırmalar

1. **Kosinüs ve Öklid eşdeğerliği.** L2-normalize vektörler için kosinüs benzerliğinin $1 - \tfrac{1}{2}\|\mathbf{x} - \mathbf{y}\|^2$'ye eşit olduğunu gösterin.
2. **BM25 ve TF-IDF.** BM25'i (Robertson ve Walker, 1994) uygulayın ve bir erişim kıyaslamasında TF-IDF ile karşılaştırın — kısa belgelerde genellikle BM25 kazanır. Neden?
3. **Kneser–Ney türetmesi.** Kneser–Ney yumuşatmasının, hiyerarşik bir Pitman–Yor sürecinin öngörü dağılımı olarak türetilebileceğini gösterin (Teh, 2006).
4. **Konu çıkarımı.** 20 Newsgroups'un bir alt kümesinde $k = 10$ ile LSA çalıştırın. 10 boyutun her biri için $|V_k|$'ya göre ilk 10 terimi inceleyin. Tanınabilir konular ortaya çıkıyor mu?

---

## Gelecek Hafta

Hafta 4 — Dağıtık Sözcük Temsilleri. Seyrek sayımlardan yoğun vektörlere: word2vec, GloVe, fastText.